# 05 · SHAP Explainability Analysis
Uses TreeExplainer to interpret the diabetes and hypertension XGBoost models.

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution value
to a specific prediction, based on game-theoretic Shapley values.

**Visualisations:**
1. Global feature importance (mean |SHAP|)
2. Beeswarm / summary plot
3. Dependence plots for top features
4. Individual prediction waterfall charts
5. Comparing SHAP vs XGBoost built-in importance

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.ml.data.diabetes_preprocessor import DiabetesPreprocessor
from src.ml.data.hypertension_preprocessor import HypertensionPreprocessor

sns.set_theme(style='white')
shap.initjs()

MODELS_DIR = Path('../models/saved')
DATA_DIR   = '../data/raw/2017_2018'
print('Ready.')

## 1 · Load models and hold-out test sets

In [ ]:
def load_model(prefix):
    files = sorted(MODELS_DIR.glob(f'{prefix}_model_*.joblib'))
    return joblib.load(files[-1]) if files else None

# Diabetes
X_d, y_d = DiabetesPreprocessor().load_and_preprocess(DATA_DIR)
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_d, y_d, test_size=0.2, stratify=y_d, random_state=42)
diab_model = load_model('diabetes')

# Hypertension
X_h, y_h = HypertensionPreprocessor().load_and_preprocess(DATA_DIR)
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, stratify=y_h, random_state=42)
htn_model = load_model('hypertension')

print(f'Diabetes test:     {X_d_test.shape}')
print(f'Hypertension test: {X_h_test.shape}')

## 2 · SHAP TreeExplainer — diabetes model

In [ ]:
if diab_model is not None:
    # Use a background sample for expected value computation
    bg = shap.sample(X_d_train, 200, random_state=42)
    explainer_d = shap.TreeExplainer(diab_model, bg)
    shap_vals_d = explainer_d.shap_values(X_d_test)
    print(f'SHAP values shape: {np.array(shap_vals_d).shape}')
    print(f'Expected value (base rate): {explainer_d.expected_value:.4f}')

## 3 · Global feature importance — mean |SHAP|

In [ ]:
if diab_model is not None:
    sv = shap_vals_d if isinstance(shap_vals_d, np.ndarray) else shap_vals_d[1]
    mean_shap = pd.Series(
        np.abs(sv).mean(axis=0),
        index=X_d_test.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 7))
    mean_shap.head(20).plot(kind='barh', color='steelblue', ax=ax)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Diabetes model — Global feature importance (mean |SHAP|)')
    plt.tight_layout()
    plt.show()

    print('Top 5 features:')
    print(mean_shap.head())

## 4 · SHAP beeswarm (summary) plot

In [ ]:
if diab_model is not None:
    sv = shap_vals_d if isinstance(shap_vals_d, np.ndarray) else shap_vals_d[1]
    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        sv, X_d_test,
        plot_type='dot',
        max_display=20,
        show=False,
        plot_size=(10, 7),
    )
    plt.title('Diabetes model — SHAP beeswarm plot')
    plt.tight_layout()
    plt.show()

## 5 · SHAP dependence plots — top 2 features

In [ ]:
if diab_model is not None:
    sv = shap_vals_d if isinstance(shap_vals_d, np.ndarray) else shap_vals_d[1]
    top_features = mean_shap.head(2).index.tolist()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, feat in zip(axes, top_features):
        shap.dependence_plot(
            feat, sv, X_d_test,
            ax=ax,
            show=False,
            dot_size=10,
        )
        ax.set_title(f'SHAP dependence: {feat}')
    plt.suptitle('Diabetes model — SHAP dependence plots', y=1.02)
    plt.tight_layout()
    plt.show()

## 6 · Individual prediction waterfall charts

In [ ]:
if diab_model is not None:
    sv = shap_vals_d if isinstance(shap_vals_d, np.ndarray) else shap_vals_d[1]
    # Pick one high-risk and one low-risk individual
    probs = diab_model.predict_proba(X_d_test)[:, 1]
    high_idx = int(probs.argmax())
    low_idx  = int(probs.argmin())

    for label, idx in [('Highest risk individual', high_idx), ('Lowest risk individual', low_idx)]:
        print(f'\n{label} — predicted probability: {probs[idx]:.4f}')
        exp = shap.Explanation(
            values=sv[idx],
            base_values=explainer_d.expected_value,
            data=X_d_test.iloc[idx].values,
            feature_names=X_d_test.columns.tolist(),
        )
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(exp, max_display=12, show=False)
        plt.title(f'{label} — SHAP waterfall')
        plt.tight_layout()
        plt.show()

## 7 · SHAP vs XGBoost built-in importance comparison

In [ ]:
if diab_model is not None:
    sv = shap_vals_d if isinstance(shap_vals_d, np.ndarray) else shap_vals_d[1]

    # XGBoost gain importance
    booster   = diab_model.get_booster()
    xgb_imp   = pd.Series(booster.get_score(importance_type='gain'))
    xgb_imp   = xgb_imp.reindex(X_d_test.columns, fill_value=0).sort_values(ascending=False)
    xgb_norm  = xgb_imp / xgb_imp.sum()

    shap_norm = mean_shap / mean_shap.sum()

    compare = pd.DataFrame({'SHAP': shap_norm, 'XGBoost gain': xgb_norm}).head(15).sort_values('SHAP')

    fig, ax = plt.subplots(figsize=(9, 7))
    compare.plot(kind='barh', ax=ax)
    ax.set_xlabel('Normalised importance')
    ax.set_title('SHAP vs XGBoost built-in importance (top 15 features)')
    plt.tight_layout()
    plt.show()

    print('Spearman rank correlation:', compare['SHAP'].corr(compare['XGBoost gain'], method='spearman').round(4))

## 8 · Hypertension model — global importance

In [ ]:
if htn_model is not None:
    bg_h = shap.sample(X_h_train, 200, random_state=42)
    explainer_h = shap.TreeExplainer(htn_model, bg_h)
    shap_vals_h = explainer_h.shap_values(X_h_test)
    sv_h = shap_vals_h if isinstance(shap_vals_h, np.ndarray) else shap_vals_h[1]

    mean_shap_h = pd.Series(
        np.abs(sv_h).mean(axis=0),
        index=X_h_test.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    mean_shap_h.head(20).plot(kind='barh', color='coral', ax=ax)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Hypertension model — Global feature importance (mean |SHAP|)')
    plt.tight_layout()
    plt.show()

    print('Top 5 features (no BP readings — preventive model):')
    print(mean_shap_h.head())

## 9 · Key SHAP findings

**Diabetes model:**
- **HbA1c** and **fasting glucose** dominate — expected, as they directly measure glycaemic control.
- **Age** and **BMI** are the next most important demographic/anthropometric features.
- **HDL cholesterol** has a *negative* SHAP effect — higher HDL reduces predicted diabetes risk.
- One-hot age groups (56-65, 65+) capture non-linear age risk better than age alone.

**Hypertension model (preventive):**
- Without BP readings, **age** becomes the dominant predictor.
- **BMI** and **waist circumference** are the most important modifiable risk factors.
- **Diabetes indicator** (HbA1c ≥ 6.5 or fasting glucose ≥ 126) is strongly associated.
- This model predicts who is *at risk of developing* hypertension — useful for prevention.

**Clinical interpretation:**
The SHAP values align well with known clinical risk factors and published guidelines
(ADA, ACC/AHA, JNC8), validating the models' clinical face validity despite being
trained on observational survey data.